# Partie 1 : Pourquoi InfluxDB ?

## SQL vs InfluxDB



### Client

On initialise les connexions aux bases de données InfluxDB et MySQL afin de permettre la lecture et l’écriture de données.

In [1]:
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS
import MySQLdb

# Config InfluxDB
URL = "http://influxdb2:8086"
TOKEN = "MyInitialAdminToken0=="
ORG = "docs"
BUCKET = "home"
MEASUREMENT = "home"

# Config MySQL
MYSQL_HOST = "mysql"
MYSQL_PORT = 3306
MYSQL_USER = "student"
MYSQL_PASSWORD = "student"
MYSQL_DATABASE = "home_data"

# Client InfluxDB
client = InfluxDBClient(url=URL, token=TOKEN, org=ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)

# Client MySQL
mysql_conn = MySQLdb.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    passwd=MYSQL_PASSWORD,
    db=MYSQL_DATABASE,
    autocommit=False,
    connect_timeout=5,
)
mysql_cursor = mysql_conn.cursor()

On définit 2 fonctions helpers pour exécuter des requêtes SQL et InfluxDB et voir leur temps d'exécution. 

In [2]:
import time

def run_sql_query(query, cursor):
  start = time.time()
  cursor.execute(query)
  result = cursor.fetchall()
  return 1000*(time.time() - start)


def run_influx_query(query, client):
  query_api = client.query_api()
  start = time.time()
  result = query_api.query(query)
  return 1000*(time.time() - start)

### Fenêtres temporelles
Premièrement, essayons de calculer la température moyenne par heure sur les 7 derniers jours.

Pour faire cela en SQL, on doit parcourir toutes les lignes, construire des groupes par heures en lisant le timestamp et ensuite agréger avec une moyenne les valeurs.

Ce genre de requêtes est typiquement bien géré par InfluxDB, autant dans la syntaxe que dans la rapidité. 

Dans ce cas, InfluxDB est plus rapide que du SQL classique car 

In [3]:
sql_query = """
SELECT
  DATE_FORMAT(time_ts, '%Y-%m-%d %H:00:00') AS hour,
  AVG(temp) AS avg_temp
FROM sensor_data
WHERE time_ts >= NOW() - INTERVAL 7 DAY
GROUP BY hour
ORDER BY hour;
"""

flux_query = '''
from(bucket: "home")
  |> range(start: -7d)
  |> filter(fn: (r) => r._measurement == "home")
  |> filter(fn: (r) => r._field == "temp")
  |> aggregateWindow(every: 1h, fn: mean, createEmpty: false)
'''

print("SQL time:", run_sql_query(sql_query, mysql_cursor), " millisecondes")
print("InfluxDB time:", run_influx_query(flux_query, client), " millisecondes")

SQL time: 116.18638038635254  millisecondes
InfluxDB time: 29.129981994628906  millisecondes


Pour les requêtes suivantes, le principe est le même sauf qu'ici on récupère la moyenne quotidienne de quantité de CO2, regroupée par étage.

Le principe est le même : faire des fenêtres temporelles est nativement plus optimisé avec InfluxDB.

In [4]:
sql_query = """
SELECT
  floor_name,
  DATE(time_ts) AS day,
  AVG(co2) AS avg_co2
FROM sensor_data
WHERE time_ts >= NOW() - INTERVAL 7 DAY
GROUP BY floor_name, day
ORDER BY floor_name, day;
"""

flux_query = '''
from(bucket: "home")
  |> range(start: -7d)
  |> filter(fn: (r) => r._measurement == "sensor_data")
  |> filter(fn: (r) => r._field == "co2")
  |> group(columns: ["floor"])
  |> aggregateWindow(every: 1d, fn: mean, createEmpty: false)
'''

print("SQL time:", run_sql_query(sql_query, mysql_cursor), " millisecondes")
print("InfluxDB time:", run_influx_query(flux_query, client), " millisecondes")

SQL time: 68.4041976928711  millisecondes
InfluxDB time: 5.942344665527344  millisecondes


### Fenêtres temporelles (variante)

Les requêtes suivantes sont des variantes des requêtes précédentes où cette fois on veut se limiter uniquement au données de la salles "Office".

Pourquoi cette variante ? Car les rooms sont stockées comme des tags dans la base de donnée et on y reviendra plus tard, mais ça permet d'optimiser les requêtes lorsqu'on filtre par tags (même si ce procédé à une contrepartie). 

Ici, l'écart entre les 2 requêtes doit être légérement plus important.

In [5]:
sql_query = """
SELECT
  DATE_FORMAT(time_ts, '%Y-%m-%d %H:00:00') AS hour,
  AVG(temp) AS avg_temp
FROM sensor_data
WHERE time_ts >= NOW() - INTERVAL 7 DAY
  AND room = 'Office'
GROUP BY hour
ORDER BY hour;
"""

flux_query = '''
from(bucket: "home")
  |> range(start: -7d)
  |> filter(fn: (r) => r._measurement == "home")
  |> filter(fn: (r) => r._field == "temp")
  |> filter(fn: (r) => r.room == "Office")
  |> aggregateWindow(every: 1h, fn: mean, createEmpty: false)
'''

print("SQL time:", run_sql_query(sql_query, mysql_cursor), " millisecondes")
print("InfluxDB time:", run_influx_query(flux_query, client), " millisecondes")

SQL time: 57.173967361450195  millisecondes
InfluxDB time: 6.749868392944336  millisecondes
